# Step 5 Part E: Improved training -- explicit turnover penalty + early stopping

Two changes from the first attempt:
1. Add an explicit penalty on turnover to the loss, not just relying on the small transaction cost already inside CVaR to discourage unnecessary trading.
2. Early stopping instead of a fixed 50 epochs, so we stop right when val performance peaks instead of training past it.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import json
from scipy.stats import norm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BTC_TRANSACTION_COST_RATE = 0.0005

train_data = np.load("train_episode_tensors.npz")
train_features = torch.tensor(train_data["features"], dtype=torch.float32)
train_spots = torch.tensor(train_data["spots"], dtype=torch.float32)
train_option_pnls = torch.tensor(train_data["option_pnls"], dtype=torch.float32)
train_masks = torch.tensor(train_data["masks"], dtype=torch.float32)

val_data = np.load("val_episode_tensors.npz")
val_features_t = torch.tensor(val_data["features"], dtype=torch.float32).to(device)
val_spots_t = torch.tensor(val_data["spots"], dtype=torch.float32).to(device)
val_option_pnls_t = torch.tensor(val_data["option_pnls"], dtype=torch.float32).to(device)
val_masks_t = torch.tensor(val_data["masks"], dtype=torch.float32).to(device)

print(f"Train: {train_features.shape[0]} episodes, Val: {val_features_t.shape[0]} episodes")

In [ ]:
class DeepHedgePolicy(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        raw_position = self.head(lstm_out)
        return 1.5 * torch.tanh(raw_position.squeeze(-1))

def simulate_pnl_batch(positions, spots, option_pnls, masks, cost_rate=BTC_TRANSACTION_COST_RATE):
    batch_size, seq_len = positions.shape
    prev_position = torch.cat([torch.zeros(batch_size, 1, device=positions.device), positions[:, :-1]], dim=1)
    trade = (positions - prev_position) * masks
    cost = trade.abs() * spots * (cost_rate / 2)
    prev_spot = torch.cat([spots[:, :1], spots[:, :-1]], dim=1)
    hedge_pnl = prev_position * (spots - prev_spot)
    total_pnl_per_step = (option_pnls + hedge_pnl - cost) * masks
    turnover = trade.abs().sum(dim=1)
    return total_pnl_per_step.sum(dim=1), turnover

def cvar_loss(terminal_pnl, alpha=0.95):
    losses = -terminal_pnl
    k = max(1, int((1 - alpha) * losses.shape[0]))
    worst_losses, _ = torch.topk(losses, k)
    return worst_losses.mean()

TURNOVER_PENALTY_WEIGHT = 5.0  # tunable -- penalizes average turnover per episode directly

def combined_loss(terminal_pnl, turnover, alpha=0.95, penalty_weight=TURNOVER_PENALTY_WEIGHT):
    return cvar_loss(terminal_pnl, alpha) + penalty_weight * turnover.mean()

## Training with early stopping

In [ ]:
model = DeepHedgePolicy(hidden_size=32).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

N_EPOCHS = 100
BATCH_SIZE = 256
PATIENCE = 10
n_train = train_features.shape[0]

train_losses, val_losses, val_cvar_only = [], [], []
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(N_EPOCHS):
    model.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    n_batches = 0

    for start in range(0, n_train, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        batch_features = train_features[idx].to(device)
        batch_spots = train_spots[idx].to(device)
        batch_option_pnls = train_option_pnls[idx].to(device)
        batch_masks = train_masks[idx].to(device)

        optimizer.zero_grad()
        positions = model(batch_features)
        terminal_pnl, turnover = simulate_pnl_batch(positions, batch_spots, batch_option_pnls, batch_masks)
        loss = combined_loss(terminal_pnl, turnover)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        val_positions = model(val_features_t)
        val_terminal_pnl, val_turnover = simulate_pnl_batch(val_positions, val_spots_t, val_option_pnls_t, val_masks_t)
        val_loss = combined_loss(val_terminal_pnl, val_turnover).item()
        val_cvar = cvar_loss(val_terminal_pnl).item()
    val_losses.append(val_loss)
    val_cvar_only.append(val_cvar)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 5 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch:3d} | train loss: {avg_train_loss:9.4f} | val loss: {val_loss:9.4f} | val CVaR only: {val_cvar:9.4f} | val turnover: {val_turnover.mean().item():.4f}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

print(f"\nBest val loss: {best_val_loss:.4f}")
torch.save(best_state, "best_deep_hedge_model_v2.pt")
print("Saved best_deep_hedge_model_v2.pt")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses, label="Train loss (CVaR + turnover penalty)")
axes[0].plot(val_losses, label="Val loss (CVaR + turnover penalty)")
axes[0].legend()
axes[0].set_title("Combined loss")

axes[1].plot(val_cvar_only, label="Val CVaR only (for comparison to v1)", color="darkred")
axes[1].legend()
axes[1].set_title("Val CVaR component alone")
plt.tight_layout()
plt.show()